# Autonomous Research Agent — LangGraph Lab

**Workflow:** Plan → Search → Read → Synthesise → Review

**The one idea to explain:** LangGraph passes a shared dictionary (the "state")
through a sequence of functions ("nodes"). Each node reads the state, does its
job, and returns the part it changed. That's it — 5 nodes, wired in a line.

| Node | Job |
|---|---|
| Plan | Topic → 3 search queries |
| Search | Tavily search for each query |
| Read | Tavily extract full article text from top URLs |
| Synthesise | LLM writes a report from the text |
| Review | LLM checks the report against sources, finalizes it |

---
## Part 1 — Install + Keys

In [1]:
!pip install langgraph langchain-groq tavily-python -q

import os
os.environ["GROQ_API_KEY"] = "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"
os.environ["TAVILY_API_KEY"] = "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"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.9 MB/s eta 0:00:00


---
## Part 2 — LLM, Search Client, and the State

`ResearchState` is the shared dictionary every node reads from and writes to.

In [2]:
from typing_extensions import TypedDict
from typing import List, Dict
from langchain_groq import ChatGroq
from tavily import TavilyClient
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser # is a small converter that extracts just the .content string from that object.
from langgraph.graph import StateGraph, START, END

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.3)
tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

class ResearchState(TypedDict):
    topic: str                  # user's original question
    search_queries: List[str]   # The 3 queries the Plan node generates
    search_results: List[Dict]  # A list of dictionaries, each one representing a search result,
    full_text: str              # All the extracted article text combined
    report: str                 # LLM's first draft of the report
    final_report: str           # Reviewed Final Version

---
## Part 3 — The 5 Nodes

Each node = one function. Reads `state`, returns a dict with the key it updated.

In [5]:
# NODE 1: Plan - turn the topic into 3 search queries
plan_chain = ChatPromptTemplate.from_template(
    "Generate exactly 3 specific search queries for this topic. One per line, no numbering.\n\nTopic: {topic}"
) | llm | StrOutputParser() # making a chain, take prompt give it to LLM , take LLM output give it to strOut..() function

def plan_node(state):
    queries = [q.strip() for q in plan_chain.invoke({"topic": state["topic"]}).split("\n") if q.strip()]

    print(f"[PLAN] {queries}") #queries is a clean list of 3 strings

    return {"search_queries": queries} # LangGraph nodes must return a dictionary containing only the key they're updating - not the entire state.


# NODE 2: Search - Tavily search for each query
def search_node(state):
    results = []
    for q in state["search_queries"]:
        for r in tavily.search(q, max_results=2)["results"]:  # send one query and asks for max 2 results
            results.append({"title": r["title"], "url": r["url"]}) # building List[Dict] - Dict {title:" " , url:""}
    print(f"[SEARCH] {len(results)} results found")
    return {"search_results": results}


# NODE 3: Read - Tavily extract full text from top 4 URLs
def read_node(state):
    urls = [r["url"] for r in state["search_results"]][:4] # A list of plain URL strings.

    extracted = tavily.extract(urls=urls)["results"] # Extracted = dict {url : "" , raw_content:" "}

    full_text = "\n\n".join(f"[{r['url']}]\n{r['raw_content'][:1500]}" for r in extracted) # "[https://anyurl.com]\nThe actual article text here..."

    print(f"[READ] Extracted {len(extracted)} articles")

    return {"full_text": full_text}


# NODE 4: Synthesise - write a report from the extracted text
synthesise_chain = ChatPromptTemplate.from_template(
    "Write a short report on '{topic}' using ONLY the source text below. "
    "Include an intro, 3 key findings, and a conclusion.\n\nSources:\n{full_text}"
) | llm | StrOutputParser()

def synthesise_node(state):
    report = synthesise_chain.invoke({"topic": state["topic"], "full_text": state["full_text"]})

    print("[SYNTHESISE] Report drafted")
    return {"report": report}


# NODE 5: Review - check the report against sources, finalize
review_chain = ChatPromptTemplate.from_template(
    "Check this report against the sources. Remove any unsupported claims, "
    "add a 'Sources' list at the end, return the final version.\n\n"
    "Sources:\n{full_text}\n\nReport:\n{report}"
) | llm | StrOutputParser()

def review_node(state):
    final = review_chain.invoke({"full_text": state["full_text"], "report": state["report"]}) #return str

    print("[REVIEW] Final report ready")
    return {"final_report": final}

---
## Part 4 — Wire the Graph and Run It

Add the 5 nodes, connect them in a straight line, compile, run.

In [4]:
graph = StateGraph(ResearchState) # Creates a new graph builder, telling it the shape of state to expect (ResearchState)

for name, fn in [("plan", plan_node), ("search", search_node), ("read", read_node),
                 ("synthesise", synthesise_node), ("review", review_node)]:
    graph.add_node(name, fn)

graph.add_edge(START, "plan")
graph.add_edge("plan", "search")
graph.add_edge("search", "read")
graph.add_edge("read", "synthesise")
graph.add_edge("synthesise", "review")
graph.add_edge("review", END)

agent = graph.compile()

result = agent.invoke({"topic": "How is AI being used in Pakistani universities?"})

print("\n\n========== FINAL REPORT ==========\n")
print(result["final_report"])

[PLAN] ['AI adoption in Pakistani higher education institutions', 'Implementation of artificial intelligence in Pakistani university research projects', 'Role of AI in enhancing student experience at Pakistani universities']
[SEARCH] 6 results found
[READ] Extracted 1 articles
[SYNTHESISE] Report drafted
[REVIEW] Final report ready


========== FINAL REPORT ==========

Report:

Introduction:
The use of Artificial Intelligence (AI) in Pakistan is a rapidly growing field, with various initiatives being implemented to facilitate progress. According to recent information, Pakistan has been actively developing its use of AI in multiple sectors since 2010, with academia and research leading the way.

Key Findings:
1. **Initiatives and Progress**: Pakistan has instituted many initiatives to facilitate the development of AI, with programmes prioritizing the development of necessary policies, research, skills, and infrastructure to disseminate AI throughout the country.
2. **Adoption Timeline**

---
## Quick Reference

| Concept | Code |
|---|---|
| State | `ResearchState` — shared dict, every node reads/writes it |
| Node | A function: `(state) -> dict` |
| Edge | `add_edge("a", "b")` — run order |
| Tavily Search | finds URLs + snippets |
| Tavily Extract | pulls full article text from URLs |

**Try after class:** change the topic in the last cell, or change `max_results=2` to `3`.